# Exp 01: Specify NumPy data types
NumPy's default integer datatype is 64-bit. All NumPy objects have values that do not exceed the max 8-bit or 32-bit value.  
Demonstrate the speed up from using smaller NumPy data type.

In [1]:
# standard libraries
from time import perf_counter_ns
import time

In [2]:
# external libraries
import numpy as np
import pandas as pd
import polars as pl

# custom libraries
from _run_constants import *
from part_00_file_db_utils import *
from part_00_process_functions_polars import *

# Control NumPy Data Type Usage

In [3]:
# set to True to demonstrate the speed up gained from using smaller NumPy data types
change_data_types = True

# Load Data

In [4]:
word_df, wg_df, letter_dict, char_matrix, \
    word_group_id_list, word_id_list, wchar_matrix = load_input_data(
        db_path=rc.DB_PATH, db_name=rc.DB_NAME,
        in_file_path=rc.IN_FILE_PATH, change_data_types=change_data_types)

...loading words into a dataframe...
...query execution took: 1.32 seconds...
...loading word groups into a dataframe...
...query execution took: 1.32 seconds...
...loading the letter dictionary...
...loading the char matrix...
...subsetting the char matrix...


In [5]:
word_df = pl.DataFrame(data = word_df)
wg_df = pl.DataFrame(data = wg_df)

In [6]:
word_df.head()

word,lcase,n_chars,first_letter,word_id,word_group_id,letter_group,letter_group_ranked
str,str,i64,str,i64,i64,str,str
"""A""","""a""",1,"""a""",0,0,"""a""","""a"""
"""aa""","""aa""",2,"""a""",1,1,"""a""","""a"""
"""aal""","""aal""",3,"""a""",2,2,"""al""","""la"""
"""aalii""","""aalii""",5,"""a""",3,3,"""ail""","""lai"""
"""aam""","""aam""",3,"""a""",4,4,"""am""","""ma"""


In [7]:
wg_df.head()

word,lcase,n_chars,first_letter,word_id,word_group_id,letter_group,letter_group_ranked,word_group_count
str,str,i64,str,i64,i64,str,str,i64
"""A""","""a""",1,"""a""",0,0,"""a""","""a""",1
"""aa""","""aa""",2,"""a""",1,1,"""a""","""a""",1
"""aal""","""aal""",3,"""a""",2,2,"""al""","""la""",2
"""aalii""","""aalii""",5,"""a""",3,3,"""ail""","""lai""",1
"""aam""","""aam""",3,"""a""",4,4,"""am""","""ma""",2


In [8]:
word_df.columns

['word',
 'lcase',
 'n_chars',
 'first_letter',
 'word_id',
 'word_group_id',
 'letter_group',
 'letter_group_ranked']

In [9]:
col_names = ['letter_selector_mod', 'n_records']
wg_df, ls_df = build_letter_selector_df(df = wg_df, ls_nchar=3,                          
                                 letter_selector_col_name='letter_selector',
                                 letter_selector_id_col_name='letter_selector_id')

In [10]:
# build a dataframe with the letter selectors
ls_df = get_ls_index(df = ls_df)


...loading the letter dictionary...


In [11]:
ls_df.head()

letter_selector_id,letter_selector,ls_count,ls_nchar_iter,ls_nchar,ls_index
u32,str,i32,i32,u32,object
0,"""xdu""",113,3,3,[False False False True False False False False False False False False False False False False False False False False True False False True False False]
1,"""kmu""",167,3,3,[False False False False False False False False False False True False True False False False False False False False True False False False False False]
2,"""jzp""",7,3,3,[False False False False False False False False False True False False False False False True False False False False False False False False False True]
3,"""xnr""",7,3,3,[False False False False False False False False False False False False False True False False False True False False False False False True False False]
4,"""wfa""",3,3,3,[ True False False False False True False False False False False False False False False False False False False False False False True False False False]


In [12]:
# should total 2387
ls_df['ls_nchar'].value_counts()


ls_nchar,count
u32,u32
1,26
2,111
3,2250


In [13]:
# this is the count of lookups for each letter selector
ls_df['ls_count'].describe()

statistic,value
str,f64
"""count""",2387.0
"""null_count""",0.0
"""mean""",90.423963
"""std""",212.422095
"""min""",1.0
"""25%""",3.0
"""50%""",14.0
"""75%""",76.0
"""max""",2544.0


In [14]:
# how many are at 99-percent?
np.quantile(a = ls_df['ls_count'], q = .99)

np.float64(1038.5199999999977)

In [15]:
# load the total number of anagrams
n_possible_anagrams = load_possible_anagrams(db_path=rc.DB_PATH,
                                             db_name=rc.DB_NAME)

...query execution took: 0.01 seconds...


In [16]:
wg_df.head()

word,lcase,n_chars,first_letter,word_id,word_group_id,letter_group,letter_group_ranked,word_group_count,letter_selector,n_records
str,str,i64,str,i64,i64,str,str,i64,str,i32
"""A""","""a""",1,"""a""",0,0,"""a""","""a""",1,"""a""",1
"""aa""","""aa""",2,"""a""",1,1,"""a""","""a""",1,"""a""",1
"""aal""","""aal""",3,"""a""",2,2,"""al""","""la""",2,"""la""",1
"""aalii""","""aalii""",5,"""a""",3,3,"""ail""","""lai""",1,"""lai""",1
"""aam""","""aam""",3,"""a""",4,4,"""am""","""ma""",2,"""ma""",1


In [17]:
# merge to identify each word's letter_selector
col_names = ["letter_selector", "letter_selector_id"]

wg_df = wg_df.join(
    ls_df.select(col_names),
    on="letter_selector"
)

In [18]:
wg_df.head()

word,lcase,n_chars,first_letter,word_id,word_group_id,letter_group,letter_group_ranked,word_group_count,letter_selector,n_records,letter_selector_id
str,str,i64,str,i64,i64,str,str,i64,str,i32,u32
"""A""","""a""",1,"""a""",0,0,"""a""","""a""",1,"""a""",1,2360
"""aa""","""aa""",2,"""a""",1,1,"""a""","""a""",1,"""a""",1,2360
"""aal""","""aal""",3,"""a""",2,2,"""al""","""la""",2,"""la""",1,1459
"""aalii""","""aalii""",5,"""a""",3,3,"""ail""","""lai""",1,"""lai""",1,996
"""aam""","""aam""",3,"""a""",4,4,"""am""","""ma""",2,"""ma""",1,702


In [19]:
ls_id_wg_id, ls_index_array = build_ls_index_arrays(wg_df=wg_df, ls_df = ls_df,change_data_types=change_data_types)

In [20]:
wg_df['letter_selector'].unique().shape

(2387,)

In [21]:
# what are the max values of the NumPy integer DataTypes
print(np.iinfo(np.int8))
print(np.iinfo(np.int16))
print(np.iinfo(np.int32))
print(np.iinfo(np.int64))

Machine parameters for int8
---------------------------------------------------------------
min = -128
max = 127
---------------------------------------------------------------

Machine parameters for int16
---------------------------------------------------------------
min = -32768
max = 32767
---------------------------------------------------------------

Machine parameters for int32
---------------------------------------------------------------
min = -2147483648
max = 2147483647
---------------------------------------------------------------

Machine parameters for int64
---------------------------------------------------------------
min = -9223372036854775808
max = 9223372036854775807
---------------------------------------------------------------



In [22]:
# check the max value of the wchar_matrix
print(wchar_matrix.dtype)
print(wchar_matrix.max())

int8
8


In [23]:
# max value of the word_group_id_list
print(word_group_id_list.dtype)
print(word_group_id_list.max())

int32
215841


In [24]:
# the only two numpy arrays in use in the code below - not created when
# the code runs - are the wchar_matrix and the word_group_id_list
# convert the wchar_matrix to int8 
# convert the word_group_id_list to int32

In [25]:
# run it!
run_start_time=perf_counter_ns()
# create the output list
output_list = np.full(shape = (n_possible_anagrams , 2), fill_value=-1, dtype=np.int32)
output_time_list = np.zeros(shape = (ls_index_array.shape[0], 3), dtype = np.float32)

# start counting
anagram_pair_count = 0

for ls_row_id, ls_row in enumerate(ls_index_array):    
        
    if ls_row_id % 100 == 0:
        print(ls_row_id)
    start_time = perf_counter_ns()
        
    ##
    # SUBSET THE wchar_matrix by column selector
    ##    
    outcome_indices = np.all(wchar_matrix[:, ls_row] >= 1, axis=1)
    
    # sub-matrix that we will use to find parent words
    ls_wchar_matrix = wchar_matrix[outcome_indices, :]
            
    # this is the list of word group ids that correspond to the word group ids
    # in the ls_wchar_matrix
    temp_wg_id_list = word_group_id_list[outcome_indices]
        
    # this is the number of word groups that meet certain criteria. 
    # for example, words that feature the letters: 'buc'    
    n_search_space = temp_wg_id_list.shape[0]        
    
    # the current list of words featuring the set of least common letters.
    # these are the words have the least common letters of 'buc'        
    curr_wg_id_list = ls_id_wg_id[ls_id_wg_id[:, 0] == ls_row_id, 1]
    # with a three-letter letter selector, ranges in size from 1 to 2544
         
    for i_curr_wg_id, curr_wg_id in enumerate(curr_wg_id_list):    
            
        # get the re-alignment of the word group id
        temp_wg_id = np.where(temp_wg_id_list == curr_wg_id)[0][0]
        
        outcome_word_id_list = temp_wg_id_list[np.all(a = (ls_wchar_matrix - ls_wchar_matrix[temp_wg_id, :]) >= 0, axis = 1)]        
                        
        n_from_words = outcome_word_id_list.shape[0]
        
        if n_from_words > 0:
            outcome_word_id_list = format_output_list(outcome_word_id_list=outcome_word_id_list, wg_id=curr_wg_id)
                                    
            # enumerate the from/parent words
            new_anagram_pair_count = anagram_pair_count + n_from_words
            
            output_list[anagram_pair_count:new_anagram_pair_count, :] = outcome_word_id_list
            
            # update the anagram pair count
            anagram_pair_count = new_anagram_pair_count

    curr_time = calc_time(time_start=start_time, round_digits=8)
    output_time_list[ls_row_id, :] = [ls_row_id, n_search_space, curr_time]

print('...time to find parent/child word relationships')
time_proc = calc_time(time_start=run_start_time, round_digits=4)
compute_elapsed_time(seconds=time_proc)
print('...truncating output list...')
output_indices = np.all(output_list >= 0, axis=1)
output_list = output_list[output_indices,]
print(output_list.shape)
time_proc = calc_time(time_start=run_start_time, round_digits=4)
compute_elapsed_time(seconds=time_proc)

0
100
200
300
400
500
600
700
800
900
1000
1100
1200
1300
1400
1500
1600
1700
1800
1900
2000
2100
2200
2300
...time to find parent/child word relationships
Hours: 0 | minutes: 1 | seconds: 28.5433
...truncating output list...
(73218235, 2)
Hours: 0 | minutes: 1 | seconds: 34.5815


In [ ]:
# current technique: 1 minute, 25 seconds
# setting the numpy data types: a little less than 34 seconds
# a full minute and change faster just by being explicit with the data types.
# this is the new standard!

In [ ]:
# count using numpy, and then create a Counter object
from_word_counter, to_word_counter = build_counters(output_list=output_list)
# this used to take 45 seconds, it now takes 6

In [ ]:
# the number of from word groups: should be 26
print(from_word_counter[746]) # should be 26
print(to_word_counter[746]) # should be 329

In [ ]:
time_df = build_timing_and_output_objects(output_time_list=output_time_list,
                                          ls_df = ls_df)

In [ ]:
# drop the ls_index field
time_df = time_df.drop(labels = ['ls_index'], axis = 1)

In [ ]:
# compute the total number of comps
time_df['total_comps'] = time_df['n_search_space'] * time_df['ls_count']

In [ ]:
# change dtypes
col_names = ['letter_selector_id',
 'n_search_space', 
 'total_comps']
for cn in col_names:
    time_df[cn] = time_df[cn].astype(int)


In [ ]:
time_df['total_time'].sum()

In [ ]:
# let's save this experiment for later use
write_data_to_sqlite(df = time_df, table_name = "exp_01_mod_meo_5", db_path = rc.DB_PATH, db_name = rc.DB_NAME)

In [ ]:
time_df.describe()

In [ ]:
time_df.loc[time_df['total_time'] >= 1, :]